# Task 3 — NLP Analysis on Extracted Well Report Text

This notebook completes **Task 3** from the PDF task description.

It uses the SQLite database created in Task 1 and applies NLP to:
- operation remarks / operation free text
- summary sections
- equipment failure remarks
- optional other free-text sections extracted in Task 1

Outputs:
- `nlp_text_sources`
- `nlp_entities`
- `nlp_operation_activity_tags`
- `nlp_report_keywords`
- summary/QC tables
- CSV exports
- an updated SQLite database containing the original Task 1 tables plus the Task 3 NLP tables


## What this notebook does

Task 3 requires three NLP components:

1. **Named Entity Recognition / Entity Extraction**
   - depths
   - equipment names
   - measurements
   - time references

2. **Activity Classification / Tagging**
   - normalized operation labels such as `TRIP_IN`, `TRIP_OUT`, `CUT`, `CEMENT`, `PRESSURE_TEST`, `EQUIPMENT_FAILURE`, `REPAIR`, `WAIT`, `CIRCULATE`, and `FISHING`

3. **TF-IDF Keyword Extraction**
   - top keywords per report, showing what makes each report unique compared with the full corpus

The implementation is intentionally rule-based and explainable because the assignment allows a rule-based classifier and domain-specific entity extraction.


## 1. Install packages

In [17]:
# Install dependencies if needed.
# In Google Colab, pandas and scikit-learn are usually already installed.
!pip -q install pandas scikit-learn tqdm


## 2. Load the Task 1 SQLite database

In [18]:
from pathlib import Path
import re
import shutil
import sqlite3
import zipfile

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

DB_PATH = Path("well_reports.sqlite")

if not DB_PATH.exists():
    try:
        from google.colab import files
        print("Upload your Task 1 SQLite database, e.g. well_reports.sqlite")
        uploaded = files.upload()
        sqlite_files = [name for name in uploaded.keys() if name.lower().endswith((".sqlite", ".db"))]
        if not sqlite_files:
            raise FileNotFoundError("No .sqlite or .db file was uploaded.")
        DB_PATH = Path(sqlite_files[0])
    except Exception as e:
        raise FileNotFoundError(
            "Could not find well_reports.sqlite. Put your Task 1 SQLite file in the notebook folder "
            "or upload it when prompted."
        ) from e

OUTPUT_DIR = Path("task3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_SQLITE = OUTPUT_DIR / "well_reports_task3_nlp.sqlite"
shutil.copy2(DB_PATH, OUTPUT_SQLITE)

print("Input database:", DB_PATH)
print("Output database copy:", OUTPUT_SQLITE)


Upload your Task 1 SQLite database, e.g. well_reports.sqlite


Saving well_reports (2).sqlite to well_reports (2) (1).sqlite
Input database: well_reports (2) (1).sqlite
Output database copy: task3_outputs/well_reports_task3_nlp.sqlite


## 3. Helper functions

In [19]:
def clean_text(x):
    """Clean OCR/PDF extracted text without changing the domain meaning."""
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\\n", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def table_exists(con, table_name):
    q = "SELECT name FROM sqlite_master WHERE type='table' AND name=?"
    return con.execute(q, (table_name,)).fetchone() is not None


def show_tables(db_path):
    with sqlite3.connect(db_path) as con:
        return pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
            con
        )


show_tables(OUTPUT_SQLITE)


,name
0,drilling_fluids
1,equipment_failures
2,extraction_errors
3,nlp_text_sources
4,operations
5,other_sections
6,report_depths
7,reports
8,sqlite_sequence
9,summary_sections


## 4. Build central NLP text-source table

In [20]:
def build_text_sources(db_path):
    """
    Create one central NLP input table from the free-text fields required by Task 3.

    Important schema note:
    In this Task 1 database, many operation remarks are stored in the `state`
    column while the `remark` column is empty. The code chooses the longer
    free-text-like field automatically.
    """
    con = sqlite3.connect(db_path)
    sources = []

    # 1) Operation remarks / operation free text
    if table_exists(con, "operations"):
        ops = pd.read_sql("""
            SELECT
                o.id AS source_record_id,
                o.report_id,
                r.wellbore_id,
                r.filename,
                o.start_time,
                o.end_time,
                o.end_depth_mmd,
                o.main_activity,
                o.sub_activity,
                o.state,
                o.remark
            FROM operations o
            LEFT JOIN reports r ON r.id = o.report_id
        """, con)

        for _, row in ops.iterrows():
            candidates = [
                clean_text(row.get("remark")),
                clean_text(row.get("state")),
            ]
            actual_remark = max(candidates, key=len)

            # If no actual free text exists, fall back to the activity fields.
            text = actual_remark if len(actual_remark) >= 3 else " ".join([
                clean_text(row.get("main_activity")),
                clean_text(row.get("sub_activity")),
                clean_text(row.get("state")),
                clean_text(row.get("remark")),
            ]).strip()

            if text:
                sources.append({
                    "source_id": f"OP_{int(row['source_record_id'])}",
                    "source_type": "operation_remark",
                    "source_record_id": int(row["source_record_id"]),
                    "report_id": row["report_id"],
                    "wellbore_id": row.get("wellbore_id"),
                    "filename": row.get("filename"),
                    "start_time": row.get("start_time"),
                    "end_time": row.get("end_time"),
                    "end_depth_mmd": row.get("end_depth_mmd"),
                    "main_activity": clean_text(row.get("main_activity")),
                    "sub_activity": clean_text(row.get("sub_activity")),
                    "operation_state": clean_text(row.get("state")),
                    "text": text,
                    "raw_remark_field": clean_text(row.get("remark")),
                    "raw_state_field": clean_text(row.get("state")),
                })

    # 2) Summary sections
    if table_exists(con, "summary_sections"):
        sums = pd.read_sql("""
            SELECT
                s.id AS source_record_id,
                s.report_id,
                r.wellbore_id,
                r.filename,
                s.section_type,
                s.section_text
            FROM summary_sections s
            LEFT JOIN reports r ON r.id = s.report_id
        """, con)

        for _, row in sums.iterrows():
            text = clean_text(row.get("section_text"))
            if text:
                sources.append({
                    "source_id": f"SUM_{int(row['source_record_id'])}",
                    "source_type": f"summary_{clean_text(row.get('section_type')).lower()}",
                    "source_record_id": int(row["source_record_id"]),
                    "report_id": row["report_id"],
                    "wellbore_id": row.get("wellbore_id"),
                    "filename": row.get("filename"),
                    "start_time": None,
                    "end_time": None,
                    "end_depth_mmd": None,
                    "main_activity": None,
                    "sub_activity": None,
                    "operation_state": None,
                    "text": text,
                    "raw_remark_field": None,
                    "raw_state_field": None,
                })

    # 3) Equipment failure remarks
    if table_exists(con, "equipment_failures"):
        fails = pd.read_sql("""
            SELECT
                e.id AS source_record_id,
                e.report_id,
                r.wellbore_id,
                r.filename,
                e.start_time,
                e.depth_mmd,
                e.depth_mtvd,
                e.equipment_system,
                e.equipment_class,
                e.downtime_minutes,
                e.remark
            FROM equipment_failures e
            LEFT JOIN reports r ON r.id = e.report_id
        """, con)

        for _, row in fails.iterrows():
            remark = clean_text(row.get("remark"))
            context = " ".join([
                clean_text(row.get("equipment_system")),
                clean_text(row.get("equipment_class")),
            ]).strip()
            text = remark if len(remark) >= 3 else context

            if text:
                sources.append({
                    "source_id": f"FAIL_{int(row['source_record_id'])}",
                    "source_type": "equipment_failure_remark",
                    "source_record_id": int(row["source_record_id"]),
                    "report_id": row["report_id"],
                    "wellbore_id": row.get("wellbore_id"),
                    "filename": row.get("filename"),
                    "start_time": row.get("start_time"),
                    "end_time": None,
                    "end_depth_mmd": row.get("depth_mmd"),
                    "main_activity": None,
                    "sub_activity": None,
                    "operation_state": None,
                    "text": " ".join([context, text]).strip(),
                    "raw_remark_field": remark,
                    "raw_state_field": None,
                })

    # 4) Optional additional free-text sections extracted by Task 1
    if table_exists(con, "other_sections"):
        other = pd.read_sql("""
            SELECT
                os.id AS source_record_id,
                os.report_id,
                r.wellbore_id,
                r.filename,
                os.section_name,
                os.section_text
            FROM other_sections os
            LEFT JOIN reports r ON r.id = os.report_id
        """, con)

        for _, row in other.iterrows():
            text = clean_text(row.get("section_text"))
            if text:
                sources.append({
                    "source_id": f"OTHER_{int(row['source_record_id'])}",
                    "source_type": f"other_{clean_text(row.get('section_name')).lower()}",
                    "source_record_id": int(row["source_record_id"]),
                    "report_id": row["report_id"],
                    "wellbore_id": row.get("wellbore_id"),
                    "filename": row.get("filename"),
                    "start_time": None,
                    "end_time": None,
                    "end_depth_mmd": None,
                    "main_activity": None,
                    "sub_activity": None,
                    "operation_state": None,
                    "text": text,
                    "raw_remark_field": None,
                    "raw_state_field": None,
                })

    con.close()

    text_sources = pd.DataFrame(sources)
    if len(text_sources):
        text_sources["text_length"] = text_sources["text"].str.len()

    return text_sources


text_sources = build_text_sources(OUTPUT_SQLITE)

print("Text sources:", text_sources.shape)
display(text_sources["source_type"].value_counts().reset_index().rename(
    columns={"index": "source_type", "source_type": "count"}
).head(20))
display(text_sources.head())


Text sources: (13266, 16)


,count,count
0,operation_remark,10983
1,summary_activities_24h,1000
2,summary_planned_activities_24h,1000
3,equipment_failure_remark,244
4,other_page 1 table 7,15
5,other_page 2 table 1,14
6,other_page 1 table 1,9
7,other_page 1 table 8,1


,source_id,source_type,source_record_id,report_id,wellbore_id,filename,start_time,end_time,end_depth_mmd,main_activity,sub_activity,operation_state,text,raw_remark_field,raw_state_field,text_length
0,OP_1,operation_remark,1,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,00:00,03:30,0.0,interruption -- other,ok,Worked to release overshot lock ring. Used C-s...,Worked to release overshot lock ring. Used C-s...,,Worked to release overshot lock ring. Used C-s...,251
1,OP_2,operation_remark,2,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,03:30,05:00,0.0,drilling -- bop/wellhea d equipment,ok,Removed tools and slings from BOP. Attached sl...,Removed tools and slings from BOP. Attached sl...,,Removed tools and slings from BOP. Attached sl...,154
2,OP_3,operation_remark,3,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,05:00,06:00,0.0,drilling -- bop/wellhea d equipment,ok,Prepared for skidding BOP. Released NT-2 conne...,Prepared for skidding BOP. Released NT-2 conne...,,Prepared for skidding BOP. Released NT-2 conne...,116
3,OP_4,operation_remark,4,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,06:00,06:30,0.0,drilling -- bop/wellhea d equipment,ok,Continued nippling down BOP. Prepared for skid...,Continued nippling down BOP. Prepared for skid...,,Continued nippling down BOP. Prepared for skid...,100
4,OP_5,operation_remark,5,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,06:30,07:15,0.0,interruption -- other,ok,Troubleshot problems with BOP trolley.,Troubleshot problems with BOP trolley.,,Troubleshot problems with BOP trolley.,38


## 5. 3a — Named Entity Recognition / entity extraction

In [21]:


EQUIPMENT_TERMS = [
    r"FLX\s+(?:packer|cutter|mandrel|assembly|assy)",
    r"ASV\s+packer",
    r"spear\s+BHA",
    r"cutter\s+BHA",
    r"fishing\s+BHA",
    r"BOP(?:\s+trolley|\s+stack)?",
    r"TDS",
    r"XO",
    r"HTS(?:\s+tailing\s+arm)?",
    r"FMS",
    r"DHSV",
    r"GLV",
    r"HPDR",
    r"PRS",
    r"LCM",
    r"OBM",
    r"HWDP",
    r"DP",
    r"overshot",
    r"Claxton\s+tool",
    r"masterbushing",
    r"mud\s+bucket",
    r"tubing\s+tong",
    r"elevator",
    r"jar",
    r"grapple",
    r"rotary",
    r"standpipe",
    r"kill\s+line",
    r"dry\s+additive\s+system",
    r"casing",
    r"drillstring",
    r"drillpipe",
    r"pup\s+joint",
    r"control\s+line",
]

ENTITY_PATTERNS = {
    "DEPTH_EXPLICIT": [
        r"\b\d{1,5}(?:[.,]\d+)?\s*(?:m\s*)?(?:mMD|mTVD|MD|TVD)\b",
    ],
    "MEASUREMENT": [
        r"\b\d+(?:[.,]\d+)?\s*(?:rpm|bar|lpm|gpm|psi|kpa|mpa|MT|ton|tons|kNm|Nm|ft\s*lbs|m3|m³|sg|ppg|ltr/min|l/min)\b",
        r"\b\d+(?:[.,]\d+)?\s*/\s*\d+(?:[.,]\d+)?\s*(?:liter|litre|ltr|lpm|bar|ton|MT|kNm)\b",
        r"\b\d+(?:[.,]\d+)?\s*(?:m/hr|m/hrs|m\s*/\s*hr|m\s*/\s*hrs)\b",
    ],
    "TIME_REFERENCE": [
        r"\b(?:[01]?\d|2[0-3]):[0-5]\d\b",
        r"\b\d+(?:[.,]\d+)?\s*(?:min|mins|minute|minutes|hr|hrs|hour|hours|h)\b",
        r"\b(?:24h|24\s*h|12h|12\s*h)\b",
    ],
    "EQUIPMENT": EQUIPMENT_TERMS,
}

compiled_patterns = {
    entity_type: [re.compile(pattern, flags=re.IGNORECASE) for pattern in patterns]
    for entity_type, patterns in ENTITY_PATTERNS.items()
}

depth_candidate_re = re.compile(r"\b\d{1,5}(?:[.,]\d+)?\s*m\b", re.I)
depth_context_keywords = re.compile(
    r"\b(rih|pooh|md|tvd|depth|tagged|casing|tubing|drillpipe|drillstring|bha|fish|from|to|at|td|spaced|pulled|ran)\b",
    re.I
)
reject_meter_context = re.compile(r"(wind|m\s*/\s*s|m/s|m\s*/\s*hr|m/hr|m/hrs)", re.I)


def extract_entities_from_text(source_id, text):
    rows = []
    seen = set()

    # Direct regex patterns
    for entity_type, patterns in compiled_patterns.items():
        out_type = "DEPTH" if entity_type == "DEPTH_EXPLICIT" else entity_type

        for pattern in patterns:
            for match in pattern.finditer(text):
                entity = match.group(0).strip(" ,.;:()[]")
                key = (out_type, match.start(), match.end(), entity.lower())

                if len(entity) >= 2 and key not in seen:
                    seen.add(key)
                    rows.append({
                        "source_id": source_id,
                        "entity_text": entity,
                        "entity_type": out_type,
                        "start_char": match.start(),
                        "end_char": match.end(),
                    })

    # Validated meter depths without explicit MD/TVD
    for match in depth_candidate_re.finditer(text):
        span = (match.start(), match.end())

        # Skip if it overlaps an explicit depth already found.
        overlaps_existing_depth = any(
            r["entity_type"] == "DEPTH"
            and not (span[1] <= r["start_char"] or span[0] >= r["end_char"])
            for r in rows
        )
        if overlaps_existing_depth:
            continue

        before = text[max(0, match.start() - 45):match.start()]
        after = text[match.end():min(len(text), match.end() + 45)]
        context = before + match.group(0) + after

        if depth_context_keywords.search(context) and not reject_meter_context.search(context):
            entity = match.group(0).strip(" ,.;:()[]")
            key = ("DEPTH", match.start(), match.end(), entity.lower())

            if key not in seen:
                seen.add(key)
                rows.append({
                    "source_id": source_id,
                    "entity_text": entity,
                    "entity_type": "DEPTH",
                    "start_char": match.start(),
                    "end_char": match.end(),
                })

    return rows


def build_entities(text_sources):
    entity_rows = []

    for _, row in text_sources.iterrows():
        for entity in extract_entities_from_text(row["source_id"], row["text"]):
            entity.update({
                "source_type": row["source_type"],
                "report_id": row["report_id"],
                "wellbore_id": row["wellbore_id"],
                "filename": row["filename"],
            })
            entity_rows.append(entity)

    entities = pd.DataFrame(entity_rows)

    if len(entities):
        columns = [
            "source_id",
            "source_type",
            "report_id",
            "wellbore_id",
            "filename",
            "entity_text",
            "entity_type",
            "start_char",
            "end_char",
        ]
        entities = entities[columns].drop_duplicates()

    return entities


entities = build_entities(text_sources)

print("Extracted entities:", entities.shape)
display(entities["entity_type"].value_counts().reset_index().rename(
    columns={"index": "entity_type", "entity_type": "count"}
))
display(entities.head(20))


Extracted entities: (81697, 9)


,count,count
0,TIME_REFERENCE,27552
1,MEASUREMENT,22487
2,DEPTH,15884
3,EQUIPMENT,15774


,source_id,source_type,report_id,wellbore_id,filename,entity_text,entity_type,start_char,end_char
0,OP_1,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,PRS,EQUIPMENT,247,250
1,OP_1,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,EQUIPMENT,18,26
2,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,30,33
3,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,EQUIPMENT,53,61
4,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,masterbushing,EQUIPMENT,115,128
5,OP_3,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,22,25
6,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,24,27
7,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,51,54
8,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP trolley,EQUIPMENT,88,99
9,OP_5,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP trolley,EQUIPMENT,26,37


## 6. 3b — Activity classification / tagging

In [22]:


activity_rules = [
    ("FISHING", [
        r"\bfish(?:ing)?\b", r"\bspear\b", r"\bovershot\b", r"\bjar\b", r"\bgrapple\b",
        r"\bstuck\b", r"\bwork(?:ed)? string free\b", r"\btagged fish\b",
    ]),
    ("CUT", [
        r"\bcut(?:ting|ter)?\b", r"\bknives\b", r"\bsever\b", r"\bsection mill\b", r"\bmill(?:ing)?\b",
    ]),
    ("CEMENT", [
        r"\bcement\b", r"\bcementing\b", r"\bcement unit\b", r"\bplug\b", r"\bshoe track\b",
    ]),
    ("PRESSURE_TEST", [
        r"\bpressure test\b", r"\bline test\b", r"\bfunction test(?:ed)?\b", r"\bFIT\b",
        r"\bleak off\b", r"\btest(?:ed)?\s+(?:to|bop|cutter|pressure)\b", r"\bpressuris",
    ]),
    ("EQUIPMENT_FAILURE", [
        r"\bfailure\b", r"\bproblem(?:s)?\b", r"\bunable\b", r"\bdamage\b", r"\bleak\b",
        r"\bplugged\b", r"\bmalfunction\b", r"\bnot able\b", r"\berratic\b", r"\bgaull?ing\b", r"\bdown\b",
    ]),
    ("REPAIR", [
        r"\brepair\b", r"\brepaired\b", r"\btroubleshot\b", r"\btroubleshoot\b", r"\bfix(?:ed)?\b",
        r"\bchanged\b", r"\breplaced\b", r"\bserviced\b", r"\bgreased\b", r"\bcleaned\b",
    ]),
    ("WAIT", [
        r"\bwait(?:ed|ing)?\b", r"\bWOC\b", r"\bstand\s*by\b", r"\bweather\b", r"\bWOW\b",
    ]),
    ("CIRCULATE", [
        r"\bcirculat", r"\bpump(?:ed|ing)?\b", r"\bflowcheck(?:ed)?\b", r"\bdisplac",
        r"\bmix(?:ed|ing)?\b", r"\breturns?\b", r"\bcondition(?:ing)?\b", r"\bLCM\b",
    ]),
    ("TRIP_OUT", [
        r"\bPOOH\b", r"\bpull(?:ed|ing)? out\b", r"\btrip out\b", r"\bout of hole\b",
    ]),
    ("TRIP_IN", [
        r"\bRIH\b", r"\brun(?:ning)? in\b", r"\btrip in\b", r"\brun in hole\b", r"\bran .*? from .*? to\b",
    ]),
]

compiled_activity_rules = [
    (tag, [re.compile(pattern, flags=re.IGNORECASE) for pattern in patterns])
    for tag, patterns in activity_rules
]


def classify_activity(text):
    matches_by_tag = {}

    for tag, patterns in compiled_activity_rules:
        hits = []

        for pattern in patterns:
            for match in pattern.finditer(text):
                hit = match.group(0).strip()
                if hit and hit.lower() not in [h.lower() for h in hits]:
                    hits.append(hit)

        if hits:
            matches_by_tag[tag] = hits

    if not matches_by_tag:
        return "OTHER", 0.20, "", ""

    # Score by number of keyword hits plus a small priority bonus from rule order.
    best_tag = None
    best_score = -1

    for idx, (tag, _) in enumerate(compiled_activity_rules):
        if tag in matches_by_tag:
            score = len(matches_by_tag[tag]) * 10 + (len(compiled_activity_rules) - idx)
            if score > best_score:
                best_score = score
                best_tag = tag

    all_tags = "; ".join(matches_by_tag.keys())
    matched_keywords = "; ".join(matches_by_tag[best_tag])
    confidence = min(
        0.95,
        0.45 + 0.10 * len(matches_by_tag[best_tag]) + 0.03 * (len(matches_by_tag) - 1)
    )

    return best_tag, round(confidence, 2), matched_keywords, all_tags


def build_activity_tags(text_sources):
    ops = text_sources[text_sources["source_type"].eq("operation_remark")].copy()
    rows = []

    for _, row in ops.iterrows():
        # Use main/sub activity as context, but keep the text column as the extracted free text.
        classify_text = " ".join([
            clean_text(row.get("main_activity")),
            clean_text(row.get("sub_activity")),
            clean_text(row["text"]),
        ])

        tag, confidence, matched_keywords, all_tags = classify_activity(classify_text)

        rows.append({
            "source_id": row["source_id"],
            "source_record_id": row["source_record_id"],
            "report_id": row["report_id"],
            "wellbore_id": row["wellbore_id"],
            "filename": row["filename"],
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "end_depth_mmd": row["end_depth_mmd"],
            "main_activity": row.get("main_activity"),
            "sub_activity": row.get("sub_activity"),
            "text": row["text"],
            "activity_tag": tag,
            "confidence": confidence,
            "matched_keywords": matched_keywords,
            "all_matched_tags": all_tags,
        })

    return pd.DataFrame(rows)


activity_tags = build_activity_tags(text_sources)

print("Tagged operation remarks:", activity_tags.shape)
display(activity_tags["activity_tag"].value_counts().reset_index().rename(
    columns={"index": "activity_tag", "activity_tag": "count"}
))
display(activity_tags.head(20))


Tagged operation remarks: (10983, 15)


,count,count
0,OTHER,3058
1,CEMENT,1729
2,CIRCULATE,1479
3,EQUIPMENT_FAILURE,1310
4,TRIP_IN,815
5,REPAIR,725
6,TRIP_OUT,587
7,CUT,390
8,FISHING,327
9,WAIT,319


,source_id,source_record_id,report_id,wellbore_id,filename,start_time,end_time,end_depth_mmd,main_activity,sub_activity,text,activity_tag,confidence,matched_keywords,all_matched_tags
0,OP_1,1,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,00:00,03:30,0.0,interruption -- other,ok,Worked to release overshot lock ring. Used C-s...,FISHING,0.55,overshot,FISHING
1,OP_2,2,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,03:30,05:00,0.0,drilling -- bop/wellhea d equipment,ok,Removed tools and slings from BOP. Attached sl...,FISHING,0.55,overshot,FISHING
2,OP_3,3,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,05:00,06:00,0.0,drilling -- bop/wellhea d equipment,ok,Prepared for skidding BOP. Released NT-2 conne...,OTHER,0.20,,
3,OP_4,4,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,06:00,06:30,0.0,drilling -- bop/wellhea d equipment,ok,Continued nippling down BOP. Prepared for skid...,EQUIPMENT_FAILURE,0.65,problems; down,EQUIPMENT_FAILURE
4,OP_5,5,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,06:30,07:15,0.0,interruption -- other,ok,Troubleshot problems with BOP trolley.,EQUIPMENT_FAILURE,0.58,problems,EQUIPMENT_FAILURE; REPAIR
5,OP_6,6,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,07:15,08:00,0.0,drilling -- bop/wellhea d equipment,ok,Skidded BOP on trolley and landed same on test...,OTHER,0.20,,
6,OP_7,7,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,08:00,08:15,0.0,drilling -- bop/wellhea d equipment,ok,Helt toolbox talk prior to running Claxton tool.,OTHER,0.20,,
7,OP_8,8,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,08:15,10:15,0.0,drilling -- bop/wellhea d equipment,ok,Picked up Claxton tool and made up to stand of...,OTHER,0.20,,
8,OP_9,9,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,10:15,14:30,0.0,drilling -- bop/wellhea d equipment,ok,Held toolbox talk. Landed Claxton tool i NT-2 ...,WAIT,0.55,Weather,WAIT
9,OP_10,10,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,14:30,00:00,0.0,interruption -- waiting on weather,ok,Watied on weather. Meanwhile worked on PRS.,WAIT,0.65,waiting; weather,WAIT


## 7. 3c — TF-IDF keyword extraction by report

In [ ]:


def build_report_keywords(text_sources, top_n=15):
    docs = (
        text_sources
        .groupby(["report_id", "wellbore_id", "filename"], dropna=False)["text"]
        .apply(lambda values: " ".join(map(str, values)))
        .reset_index()
    )

    docs["text"] = docs["text"].map(clean_text)
    docs = docs[docs["text"].str.len() > 0].reset_index(drop=True)

    if len(docs) == 0:
        return pd.DataFrame(columns=[
            "report_id", "wellbore_id", "filename", "keyword", "tfidf_score", "rank"
        ])

    domain_stopwords = set("""
        ok well wells report reports start end depth time main sub activity state remark operations operation
        table section summary planned made make used using prior mmd mtvd md tvd null none nan page
        conditioni interruptio bop/wellhea wellhea ng co oth equ th ment mpletion tion
        x000d x000a
    """.split())

    stop_words = sorted(set(ENGLISH_STOP_WORDS).union(domain_stopwords))

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words=stop_words,
        ngram_range=(1, 2),
        min_df=2 if len(docs) >= 10 else 1,
        max_df=0.90,
        max_features=8000,
       token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_/-]{1,}\b",
    )

    X = vectorizer.fit_transform(docs["text"])
    terms = np.array(vectorizer.get_feature_names_out())

    keyword_rows = []

    for i, doc_row in docs.iterrows():
        row = X.getrow(i)

        if row.nnz == 0:
            continue

        order = np.argsort(row.data)[::-1][:top_n]

        for rank, pos in enumerate(order, start=1):
            term_index = row.indices[pos]

            keyword_rows.append({
                "report_id": doc_row["report_id"],
                "wellbore_id": doc_row["wellbore_id"],
                "filename": doc_row["filename"],
                "keyword": terms[term_index],
                "tfidf_score": round(float(row.data[pos]), 6),
                "rank": rank,
            })

    return pd.DataFrame(keyword_rows)


report_keywords = build_report_keywords(text_sources, top_n=15)

print("Report keywords:", report_keywords.shape)
display(report_keywords.head(30))


Report keywords: (14935, 6)


,report_id,wellbore_id,filename,keyword,tfidf_score,rank
0,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,trolley,0.305802,1
1,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,claxton tool,0.251927,2
2,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,claxton,0.249941,3
3,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,bop,0.242190,4
4,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,nt-2,0.204114,5
5,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,lock ring,0.200644,6
6,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,0.173789,7
7,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,lock,0.151008,8
8,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,tool nt-2,0.148475,9
9,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,nt-2 connector,0.148475,10


## 8. Save SQLite tables, CSV exports, and zip file

In [24]:


entity_summary = (
    entities
    .groupby("entity_type")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

activity_summary = (
    activity_tags
    .groupby("activity_tag")
    .agg(
        count=("source_id", "count"),
        avg_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
activity_summary["avg_confidence"] = activity_summary["avg_confidence"].round(3)

source_summary = (
    text_sources
    .groupby("source_type")
    .agg(
        count=("source_id", "count"),
        avg_text_length=("text_length", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
source_summary["avg_text_length"] = source_summary["avg_text_length"].round(1)

# SQLite export. We do NOT use method="multi" to avoid SQLite variable-limit errors.
with sqlite3.connect(OUTPUT_SQLITE) as con:
    text_sources.to_sql("nlp_text_sources", con, if_exists="replace", index=False, chunksize=1000)
    entities.to_sql("nlp_entities", con, if_exists="replace", index=False, chunksize=1000)
    activity_tags.to_sql("nlp_operation_activity_tags", con, if_exists="replace", index=False, chunksize=1000)
    report_keywords.to_sql("nlp_report_keywords", con, if_exists="replace", index=False, chunksize=1000)

    entity_summary.to_sql("nlp_entity_summary", con, if_exists="replace", index=False)
    activity_summary.to_sql("nlp_activity_summary", con, if_exists="replace", index=False)
    source_summary.to_sql("nlp_source_summary", con, if_exists="replace", index=False)

# CSV exports
csv_paths = {
    "nlp_text_sources": OUTPUT_DIR / "nlp_text_sources.csv",
    "nlp_entities": OUTPUT_DIR / "nlp_entities.csv",
    "nlp_operation_activity_tags": OUTPUT_DIR / "nlp_operation_activity_tags.csv",
    "nlp_report_keywords": OUTPUT_DIR / "nlp_report_keywords.csv",
    "nlp_entity_summary": OUTPUT_DIR / "nlp_entity_summary.csv",
    "nlp_activity_summary": OUTPUT_DIR / "nlp_activity_summary.csv",
    "nlp_source_summary": OUTPUT_DIR / "nlp_source_summary.csv",
}

text_sources.to_csv(csv_paths["nlp_text_sources"], index=False)
entities.to_csv(csv_paths["nlp_entities"], index=False)
activity_tags.to_csv(csv_paths["nlp_operation_activity_tags"], index=False)
report_keywords.to_csv(csv_paths["nlp_report_keywords"], index=False)

entity_summary.to_csv(csv_paths["nlp_entity_summary"], index=False)
activity_summary.to_csv(csv_paths["nlp_activity_summary"], index=False)
source_summary.to_csv(csv_paths["nlp_source_summary"], index=False)

# Convenient wide keyword file: one row per report, keyword_rank_1...keyword_rank_15.
keywords_wide = report_keywords.pivot_table(
    index=["report_id", "wellbore_id", "filename"],
    columns="rank",
    values="keyword",
    aggfunc="first",
).reset_index()
keywords_wide.columns = [
    f"keyword_rank_{col}" if isinstance(col, int) else col
    for col in keywords_wide.columns
]
keywords_wide_path = OUTPUT_DIR / "nlp_report_keywords_wide.csv"
keywords_wide.to_csv(keywords_wide_path, index=False)

# Zip all outputs created by this notebook.
zip_path = Path("task3_nlp_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(OUTPUT_SQLITE, arcname=OUTPUT_SQLITE.name)
    for path in csv_paths.values():
        z.write(path, arcname=path.name)
    z.write(keywords_wide_path, arcname=keywords_wide_path.name)

print("Saved updated SQLite:", OUTPUT_SQLITE)
print("Saved CSV folder:", OUTPUT_DIR)
print("Saved zip:", zip_path)

display(source_summary)
display(entity_summary)
display(activity_summary)


Saved updated SQLite: task3_outputs/well_reports_task3_nlp.sqlite
Saved CSV folder: task3_outputs
Saved zip: task3_nlp_outputs.zip


,source_type,count,avg_text_length
1,operation_remark,10983,116.2
7,summary_planned_activities_24h,1000,2532.6
6,summary_activities_24h,1000,157.8
0,equipment_failure_remark,244,127.9
3,other_page 1 table 7,15,13.1
5,other_page 2 table 1,14,13.2
2,other_page 1 table 1,9,34.7
4,other_page 1 table 8,1,14.0


,entity_type,count
3,TIME_REFERENCE,27552
2,MEASUREMENT,22487
0,DEPTH,15884
1,EQUIPMENT,15774


,activity_tag,count,avg_confidence
5,OTHER,3058,0.200
0,CEMENT,1729,0.598
1,CIRCULATE,1479,0.632
3,EQUIPMENT_FAILURE,1310,0.576
8,TRIP_IN,815,0.554
7,REPAIR,725,0.576
9,TRIP_OUT,587,0.567
2,CUT,390,0.623
4,FISHING,327,0.609
10,WAIT,319,0.643


## 9. Validate saved outputs

In [26]:
# ---------------------------------------------------------------------
# Quick validation queries from the saved SQLite database
# ---------------------------------------------------------------------

with sqlite3.connect(OUTPUT_SQLITE) as con:
    saved_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'nlp_%' ORDER BY name",
        con
    )
    counts = []
    for table_name in saved_tables["name"]:
        n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table_name}", con)["n"].iloc[0]
        counts.append({"table": table_name, "rows": n})

display(pd.DataFrame(counts))

print("Example entities")
display(entities.head(10))

print("Example activity tags")
display(activity_tags[["source_id", "wellbore_id", "activity_tag", "confidence", "matched_keywords", "text"]].head(10))

print("Example report keywords")
display(report_keywords.head(20))


,table,rows
0,nlp_activity_summary,11
1,nlp_entities,81697
2,nlp_entity_summary,4
3,nlp_operation_activity_tags,10983
4,nlp_report_keywords,14935
5,nlp_source_summary,8
6,nlp_text_sources,13266


Example entities


,source_id,source_type,report_id,wellbore_id,filename,entity_text,entity_type,start_char,end_char
0,OP_1,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,PRS,EQUIPMENT,247,250
1,OP_1,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,EQUIPMENT,18,26
2,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,30,33
3,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,EQUIPMENT,53,61
4,OP_2,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,masterbushing,EQUIPMENT,115,128
5,OP_3,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,22,25
6,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,24,27
7,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP,EQUIPMENT,51,54
8,OP_4,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP trolley,EQUIPMENT,88,99
9,OP_5,operation_remark,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,BOP trolley,EQUIPMENT,26,37


Example activity tags


,source_id,wellbore_id,activity_tag,confidence,matched_keywords,text
0,OP_1,15/9-F-12,FISHING,0.55,overshot,Worked to release overshot lock ring. Used C-s...
1,OP_2,15/9-F-12,FISHING,0.55,overshot,Removed tools and slings from BOP. Attached sl...
2,OP_3,15/9-F-12,OTHER,0.20,,Prepared for skidding BOP. Released NT-2 conne...
3,OP_4,15/9-F-12,EQUIPMENT_FAILURE,0.65,problems; down,Continued nippling down BOP. Prepared for skid...
4,OP_5,15/9-F-12,EQUIPMENT_FAILURE,0.58,problems,Troubleshot problems with BOP trolley.
5,OP_6,15/9-F-12,OTHER,0.20,,Skidded BOP on trolley and landed same on test...
6,OP_7,15/9-F-12,OTHER,0.20,,Helt toolbox talk prior to running Claxton tool.
7,OP_8,15/9-F-12,OTHER,0.20,,Picked up Claxton tool and made up to stand of...
8,OP_9,15/9-F-12,WAIT,0.55,Weather,Held toolbox talk. Landed Claxton tool i NT-2 ...
9,OP_10,15/9-F-12,WAIT,0.65,waiting; weather,Watied on weather. Meanwhile worked on PRS.


Example report keywords


,report_id,wellbore_id,filename,keyword,tfidf_score,rank
0,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,trolley,0.305802,1
1,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,claxton tool,0.251927,2
2,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,claxton,0.249941,3
3,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,bop,0.242190,4
4,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,nt-2,0.204114,5
5,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,lock ring,0.200644,6
6,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,overshot,0.173789,7
7,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,lock,0.151008,8
8,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,tool nt-2,0.148475,9
9,1,15/9-F-12,15_9_F_12_2007_09_11.pdf,nt-2 connector,0.148475,10
